# 07d — FeUdal Networks (Reference Import) + Optuna TPE

**Objectiu**: Importar la implementació de referència de FeUdal Networks (Vezhnevets et al. 2017) des de `references/feudalnets-pytorch-master/` amb adaptacions mínimes per al trading de BTC/USD.

**Adaptacions**:
- Perception module: CNN → configurable MLP per a observacions 1D (71 dims)
- VecEnv wrapper per a `CryptoMarketEnv`
- Preprocessor: identity (data ja normalitzada)
- Optimització d'hiperparàmetres via Optuna TPE (15 HPs)

**Referències**:
- Vezhnevets et al. (2017): FeUdal Networks for Hierarchical Reinforcement Learning
- Reference code: https://github.com/dmakian/feudalnets-pytorch

In [ ]:
# CEL·LA 1 — IMPORTS I PATH SETUP

import argparse
import json
import os as _os
import random
import sys
import warnings
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import optuna
import polars as pl
import torch
import torch.nn as nn
from codecarbon import EmissionsTracker
from torch.distributions import Categorical
from torch.utils.tensorboard import SummaryWriter
from tqdm.auto import tqdm

# ── gym → gymnasium compatibility shim ──
# Reference code uses old gym, but we have gymnasium installed
import gymnasium
sys.modules['gym'] = gymnasium
sys.modules['gym.wrappers'] = gymnasium.wrappers
print("✓ gym→gymnasium compatibility shim installed")

# ── CWD fix ──
_cwd = Path.cwd()
if not (_cwd / "data" / "processed").exists():
    _root = _cwd.parents[1]
    if (_root / "data" / "processed").exists():
        _os.chdir(_root)
        print(f"Changed CWD to project root: {_root}")

# ── Project imports ──
from src.data.manager import DataManager
from src.rl.envs.crypto_market_env import (
    CryptoMarketEnv, INITIAL_BALANCE, HOLD_PENALTY, steps_per_4h_for,
)
from src.rl.metrics import compute_sharpe, compute_max_drawdown
from src.rl.utils import fee_curriculum_factor

# ── Reference FeudalNet code ──
_REF_DIR = str(Path(".").resolve() / "references" / "feudalnets-pytorch-master")
if _REF_DIR not in sys.path:
    sys.path.insert(0, _REF_DIR)
    print(f"Added to sys.path: {_REF_DIR}")

from feudalnet import FeudalNetwork, feudal_loss
from storage import Storage
from utils import take_action, weight_init

warnings.filterwarnings('ignore')
optuna.logging.set_verbosity(optuna.logging.WARNING)
print('✓ Imports OK')

In [ ]:
# CEL·LA 2 — CONFIGURACIÓ

SMOKE = False  # Rapid validation (<5 min). Set False for full run.

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# ── Paths ──
RESULTS = Path("results_dense") / "07d_hrl_import"
RESULTS.mkdir(parents=True, exist_ok=True)
RUNS_DIR = RESULTS / "runs"
RUNS_DIR.mkdir(parents=True, exist_ok=True)
OPTUNA_DB_DIR = RESULTS / "optuna_db"
OPTUNA_DB_DIR.mkdir(parents=True, exist_ok=True)
CODECARBON_DIR = RESULTS / "codecarbon"
CODECARBON_DIR.mkdir(parents=True, exist_ok=True)

NB_TAG = "import"
OPTUNA_P1_JSON = RESULTS / "optuna_p1_best_params.json"

# ── Data config ──
TIMEFRAME = "60m"

# ── Fee curriculum ──
FEE_WARMUP_FRAC   = 0.40  # 0% fees fins 40% timesteps
FEE_RAMP_END_FRAC = 0.60  # 100% fees a 60% timesteps (ramp lineal 40%→60%)

# ── Default hyperparameters (from FuN paper) ──
DEFAULT_HP = {
    'lr':                0.0005,     # RMSprop learning rate
    'hidden_dim_manager': 256,       # d: manager/perception output dim
    'hidden_dim_worker':  16,        # k: worker hidden dim
    'time_horizon':       10,        # c: manager temporal horizon
    'dilation':           10,        # r: DilatedLSTM dilation level
    'gamma_w':            0.99,      # worker discount
    'gamma_m':            0.999,     # manager discount
    'alpha':              0.5,       # intrinsic reward coefficient
    'entropy_coef':       0.01,      # entropy bonus weight
    'num_steps':          400,       # rollout length before update
    'num_workers':        8,         # parallel envs (FIXED)
    'grad_clip':          5.0,       # gradient clipping norm
    'eps':                1e-5,      # random goal exploration probability
    # Custom
    'perception_hidden':  128,       # MLP intermediate hidden dim
    'hold_penalty':       -0.00002,  # env: FLAT+HOLD penalty
    'realize_bonus_scale': 0.7,      # env: close bonus scale
}

# ── Training config ──
MAX_STEPS       = 2_000_000  # Total training steps (env interactions)
EVAL_FREQ       = 50_000     # Steps between evaluations
N_EVAL_EPS      = 5          # Episodes per evaluation
N_TRIALS        = 50         # Optuna TPE trials
N_STARTUP       = 15         # Random startup trials
OPTUNA_OBJECTIVE = 'best_return'

if SMOKE:
    MAX_STEPS       = 5_000
    EVAL_FREQ       = 2_000
    N_EVAL_EPS      = 2
    N_TRIALS        = 5
    N_STARTUP       = 2
    DEFAULT_HP['num_steps'] = 50
    print("⚠️  SMOKE TEST MODE ENABLED")

print(f"Device: {DEVICE}")
print(f"Seed:   {SEED}")
print(f"Results: {RESULTS}")
print(f"Fee curriculum: warmup={FEE_WARMUP_FRAC:.0%} ramp_end={FEE_RAMP_END_FRAC:.0%}")

In [ ]:
# CEL·LA 3 — DATA

dm = DataManager(interval=TIMEFRAME, project_root=Path("."))
FEATURE_COLS = dm.features
N_FEATURES = len(FEATURE_COLS)
df_train, df_val, df_test = dm.load_all()

print(f"Train: {len(df_train):,} | Val: {len(df_val):,} | Test: {len(df_test):,}")
print(f"Features: {N_FEATURES} | OBS_DIM: {N_FEATURES + 6}")

In [ ]:
# CEL·LA 4 — ENTORN + VECENV

def make_env(df, **kwargs):
    """Factory per crear CryptoMarketEnv."""
    return CryptoMarketEnv(
        df=df,
        feature_cols=FEATURE_COLS,
        steps_per_4h=steps_per_4h_for(TIMEFRAME),
        **kwargs
    )


class SimpleVecEnv:
    """Vectorized environment for N parallel CryptoMarketEnv instances.
    
    Interface compatible amb el codi de referència:
      - reset() -> obs_batch (N, obs_dim)
      - step(actions) -> obs, rewards, dones, infos
    
    Bridges Gymnasium 5-tuple → old Gym 4-tuple.
    Auto-reset on done (reference code expects this).
    """
    def __init__(self, env_fns):
        self.envs = [fn() for fn in env_fns]
        self.num_envs = len(self.envs)
        # Gym-compatible attributes
        self.single_observation_space = self.envs[0].observation_space
        self.single_action_space = self.envs[0].action_space

    def reset(self):
        obs_list = []
        for i, env in enumerate(self.envs):
            obs, _ = env.reset(seed=SEED + i)
            obs_list.append(obs)
        return np.stack(obs_list)

    def step(self, actions):
        obs_list, reward_list, done_list, info_list = [], [], [], []
        for i, (env, a) in enumerate(zip(self.envs, actions)):
            obs, reward, terminated, truncated, info = env.step(int(a))
            done = terminated or truncated
            if done:
                # Auto-reset
                obs, _ = env.reset(seed=SEED + i + self.num_envs)
            obs_list.append(obs)
            reward_list.append(reward)
            done_list.append(done)
            # Match reference logger format
            if done:
                info['returns/episodic_reward'] = info['equity'] / INITIAL_BALANCE - 1
                info['returns/episodic_length'] = info['step']
            else:
                info['returns/episodic_reward'] = None
                info['returns/episodic_length'] = None
            info_list.append(info)
        return (np.stack(obs_list),
                np.array(reward_list),
                np.array(done_list),
                info_list)

    def close(self):
        for env in self.envs:
            env.close()

    def seed(self, s):
        pass  # Seeds handled in reset


def make_vec_env(df, n_envs, **kwargs):
    """Factory per crear VecEnv."""
    return SimpleVecEnv([lambda: make_env(df, **kwargs) for _ in range(n_envs)])


# Verify dimensions
_env = make_env(df_train)
_obs, _ = _env.reset(seed=SEED)
OBS_DIM = _env.observation_space.shape[0]
N_ACTIONS = _env.action_space.n
_env.close()
del _env

print(f"OBS_DIM={OBS_DIM} | N_ACTIONS={N_ACTIONS}")
print(f"✓ Environment factory ready")

In [ ]:
# CEL·LA 5 — CUSTOM PERCEPTION

class CustomPerception(nn.Module):
    """MLP perception for 1D feature vectors.
    
    Replaces the reference Perception module (designed for Atari images or
    fixed 64-dim hidden layer MLP).
    
    Args:
        input_dim: observation dimension (71)
        output_dim: latent dimension d (hidden_dim_manager)
        hidden_dim: intermediate hidden layer dimension (configurable)
    """
    def __init__(self, input_dim, output_dim, hidden_dim=128):
        super().__init__()
        self.percept = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, output_dim),
            nn.ReLU()
        )
    
    def forward(self, x):
        return self.percept(x)


print(f"CustomPerception: {OBS_DIM} -> hidden -> d")

In [ ]:
# CEL·LA 6 — AGENT FACTORY

def make_args(hp: dict):
    """Create argparse.Namespace mimicking reference args object.
    
    Required by FeudalNetwork.__init__ and feudal_loss.
    """
    args = argparse.Namespace(**hp)
    return args


def create_feudal_agent(hp: dict, device=DEVICE):
    """Instantiate FeudalNetwork from reference code with custom perception.
    
    Key adaptations:
    1. Swap perception module (CNN → configurable MLP)
    2. Bypass preprocessor (data already normalized)
    3. Create VecEnv with environment-specific kwargs
    
    Returns:
        Tuple (feudalnet, envs, args)
    """
    args = make_args(hp)
    
    envs = make_vec_env(
        df_train,
        hp['num_workers'],
        hold_penalty=hp.get('hold_penalty', HOLD_PENALTY),
        realize_bonus_scale=hp.get('realize_bonus_scale', 0.7)
    )
    
    feudalnet = FeudalNetwork(
        num_workers=hp['num_workers'],
        input_dim=(OBS_DIM,),  # Reference expects shape tuple
        hidden_dim_manager=hp['hidden_dim_manager'],
        hidden_dim_worker=hp['hidden_dim_worker'],
        n_actions=N_ACTIONS,
        time_horizon=hp['time_horizon'],
        dilation=hp['dilation'],
        device=device,
        mlp=True,  # Use MLP mode (not CNN)
        args=args
    )
    
    # ── Swap perception with configurable MLP ──
    perception_hidden = hp.get('perception_hidden', 128)
    feudalnet.percept = CustomPerception(
        input_dim=OBS_DIM,
        output_dim=hp['hidden_dim_manager'],
        hidden_dim=perception_hidden
    ).to(device)
    feudalnet.percept.apply(weight_init)
    
    # ── Bypass preprocessor (identity passthrough) ──
    # Data is already pre-normalized via NB01
    feudalnet.preprocessor = lambda x: torch.FloatTensor(
        np.asarray(x).reshape(x.shape[0], -1)
    ).to(device)
    
    return feudalnet, envs, args


# Quick test
_fn, _envs, _args = create_feudal_agent(DEFAULT_HP)
print(f"FeudalNetwork params: {sum(p.numel() for p in _fn.parameters()):,}")
print(f"  Perception: {sum(p.numel() for p in _fn.percept.parameters()):,}")
print(f"  Manager:    {sum(p.numel() for p in _fn.manager.parameters()):,}")
print(f"  Worker:     {sum(p.numel() for p in _fn.worker.parameters()):,}")
_envs.close()
del _fn, _envs
print("✓ Agent factory ready")

In [ ]:
# CEL·LA 7 — TRAINING LOOP

def train_feudal(hp: dict, max_steps: int, eval_freq: int = None,
                 df_tr=None, df_vl=None, trial=None,
                 writer=None, verbose=1,
                 fee_warmup_frac: float = FEE_WARMUP_FRAC,
                 fee_ramp_end_frac: float = FEE_RAMP_END_FRAC):
    """Training loop adapted from reference main.py experiment().

    Key adaptations:
    - Periodic evaluation on validation set
    - Optuna pruning integration
    - Episode tracking via info dict
    - Fee curriculum: fees progressives 0→100% entre warmup i ramp_end

    Args:
        hp: hyperparameters dict
        max_steps: total environment steps
        eval_freq: steps between evaluations (None = no eval)
        df_tr: training dataframe
        df_vl: validation dataframe (for evaluation)
        trial: optuna.Trial for pruning (None = no pruning)
        writer: TensorBoard SummaryWriter
        verbose: 0=silent, 1=progress
        fee_warmup_frac: fracció de max_steps fins a fee_factor=0
        fee_ramp_end_frac: fracció de max_steps on fee_factor arriba a 1.0

    Returns:
        dict with:
        - feudalnet: trained model
        - eval_history: list of evaluation dicts
        - episode_rewards: list of episodic rewards
    """
    if df_tr is None:
        df_tr = df_train
    if df_vl is None:
        df_vl = df_val

    device = DEVICE
    args = make_args(hp)

    feudalnet, envs, args = create_feudal_agent(hp, device)
    optimizer = torch.optim.RMSprop(feudalnet.parameters(), lr=hp['lr'],
                                     alpha=0.99, eps=1e-5)

    goals, states, masks = feudalnet.init_obj()
    x = envs.reset()

    step = 0
    episode_rewards = []
    eval_history = []
    eval_counter = 0

    pbar = tqdm(total=max_steps, disable=(verbose == 0), desc="Training")

    while step < max_steps:
        # ── Fee curriculum ──
        factor = fee_curriculum_factor(step, max_steps, fee_warmup_frac, fee_ramp_end_frac)
        for env_i in envs.envs:
            env_i.fee_factor = factor

        # Detach LSTMs and goals (truncated BPTT)
        feudalnet.repackage_hidden()
        goals = [g.detach() for g in goals]

        storage = Storage(size=hp['num_steps'],
                         keys=['r', 'r_i', 'v_w', 'v_m', 'logp', 'entropy',
                               's_goal_cos', 'mask', 'ret_w', 'ret_m',
                               'adv_m', 'adv_w'])

        # ── Rollout collection ──
        for _ in range(hp['num_steps']):
            action_dist, goals, states, value_m, value_w = \
                feudalnet(x, goals, states, masks[-1])

            action, logp, entropy = take_action(action_dist)
            x, reward, done, info = envs.step(action)

            # Log episodes
            for ep_info in info:
                if ep_info['returns/episodic_reward'] is not None:
                    episode_rewards.append(ep_info['returns/episodic_reward'])
                    if writer:
                        writer.add_scalar('returns/episodic_reward',
                                         ep_info['returns/episodic_reward'], step)

            mask = torch.FloatTensor(1 - done).unsqueeze(-1).to(device)
            masks.pop(0)
            masks.append(mask)

            storage.add({
                'r': torch.FloatTensor(reward).unsqueeze(-1).to(device),
                'r_i': feudalnet.intrinsic_reward(states, goals, masks),
                'v_w': value_w,
                'v_m': value_m,
                'logp': logp.unsqueeze(-1),
                'entropy': entropy.unsqueeze(-1),
                's_goal_cos': feudalnet.state_goal_cosine(states, goals, masks),
                'm': mask
            })

            step += hp['num_workers']

        # ── Compute next values for bootstrapping ──
        with torch.no_grad():
            *_, next_v_m, next_v_w = feudalnet(x, goals, states, mask, save=False)
            next_v_m = next_v_m.detach()
            next_v_w = next_v_w.detach()

        # ── Update ──
        optimizer.zero_grad()
        loss, loss_dict = feudal_loss(storage, next_v_m, next_v_w, args)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(feudalnet.parameters(), hp['grad_clip'])
        optimizer.step()

        if writer:
            writer.add_scalar('train/fee_factor', factor, step)
            for k, v in loss_dict.items():
                writer.add_scalar(k, v, step)

        pbar.update(hp['num_steps'] * hp['num_workers'])

        # ── Periodic evaluation ──
        if eval_freq and step >= (eval_counter + 1) * eval_freq:
            eval_counter = step // eval_freq
            eval_result = evaluate_feudal(feudalnet, df_vl, hp, n_episodes=N_EVAL_EPS)
            eval_history.append(eval_result)

            if writer:
                for k, v in eval_result.items():
                    writer.add_scalar(f'eval/{k}', v, step)

            if verbose:
                pbar.set_postfix(ret=f"{eval_result['mean_return']:.2f}%",
                                sharpe=f"{eval_result['sharpe']:.2f}",
                                fee=f"{factor:.2f}")

            # Optuna pruning
            if trial:
                trial.report(eval_result['mean_return'], eval_counter)
                if trial.should_prune():
                    envs.close()
                    pbar.close()
                    raise optuna.TrialPruned()

    pbar.close()
    envs.close()

    return {
        'feudalnet': feudalnet,
        'eval_history': eval_history,
        'episode_rewards': episode_rewards,
    }


print("✓ Training loop ready (fee curriculum integrat)")

In [ ]:
# CEL·LA 8 — EVALUATION

def evaluate_feudal(feudalnet, df, hp, n_episodes=5, deterministic=True):
    """Evaluate FeudalNetwork on validation/test set.
    
    Runs single-env sequential episodes (not vectorized) for clean evaluation.
    
    Args:
        feudalnet: trained model
        df: evaluation dataframe
        hp: hyperparameters dict
        n_episodes: number of episodes to run
        deterministic: if True, use argmax; else sample from distribution
        
    Returns:
        dict with mean_return, std_return, sharpe, max_drawdown, mean_trades, position_frac
    """
    env = make_env(df, hold_penalty=hp.get('hold_penalty', HOLD_PENALTY),
                   realize_bonus_scale=hp.get('realize_bonus_scale', 0.7))
    feudalnet.eval()
    
    all_returns, all_equities, all_trades, all_pos_sides = [], [], [], []
    rng = np.random.default_rng(SEED)
    
    # Temporarily change internal batch size to 1
    original_b = feudalnet.b
    
    with torch.no_grad():
        for ep in range(n_episodes):
            obs, info = env.reset(seed=int(rng.integers(0, 1_000_000)))
            ep_eq = [info['equity']]
            done = False
            
            # Initialize single-worker hidden states and goal/state lists
            d = hp['hidden_dim_manager']
            c = hp['time_horizon']
            r = hp['dilation']
            k = hp['hidden_dim_worker']
            
            hidden_m = (torch.zeros(1, r * d).to(DEVICE),
                       torch.zeros(1, r * d).to(DEVICE))
            hidden_w = (torch.zeros(1, k * N_ACTIONS).to(DEVICE),
                       torch.zeros(1, k * N_ACTIONS).to(DEVICE))
            goals = [torch.zeros(1, d).to(DEVICE) for _ in range(2*c+1)]
            states = [torch.zeros(1, d).to(DEVICE) for _ in range(2*c+1)]
            masks = [torch.ones(1, 1).to(DEVICE) for _ in range(2*c+1)]
            
            # Temporarily set batch to 1
            feudalnet.b = 1
            feudalnet.hidden_m = hidden_m
            feudalnet.hidden_w = hidden_w
            feudalnet.worker.b = 1
            
            while not done:
                x_t = obs.reshape(1, -1)  # (1, obs_dim)
                
                action_dist, goals, states, _, _ = feudalnet(
                    x_t, goals, states, masks[-1])
                
                if deterministic:
                    action = action_dist.argmax(dim=-1).item()
                else:
                    action = Categorical(action_dist).sample().item()
                
                obs, _, term, trunc, info = env.step(action)
                done = term or trunc
                
                mask = torch.FloatTensor([0.0 if done else 1.0]).unsqueeze(-1).to(DEVICE)
                masks.pop(0)
                masks.append(mask)
                
                ep_eq.append(info['equity'])
                all_pos_sides.append(info['pos_side'])
            
            all_returns.append((ep_eq[-1] / ep_eq[0] - 1) * 100)
            all_equities.append(ep_eq)
            all_trades.append(info['n_trades'])
    
    # Restore original batch size
    feudalnet.b = original_b
    feudalnet.worker.b = original_b
    feudalnet.train()
    env.close()
    
    n_total = len(all_pos_sides)
    position_frac = sum(1 for p in all_pos_sides if p != 0) / max(n_total, 1)
    
    return {
        'mean_return': float(np.mean(all_returns)),
        'std_return': float(np.std(all_returns)),
        'sharpe': float(np.mean([compute_sharpe(eq) for eq in all_equities])),
        'max_drawdown': float(compute_max_drawdown(
            [e for eq in all_equities for e in eq])),
        'mean_trades': float(np.mean(all_trades)),
        'position_frac': position_frac,
    }


print("✓ Evaluation function ready")

In [ ]:
# CEL·LA 9 — SMOKE VALIDATION

writer_smoke = SummaryWriter(log_dir=str(RUNS_DIR / 'smoke_test'))

result_smoke = train_feudal(
    hp=DEFAULT_HP,
    max_steps=50_000,
    eval_freq=EVAL_FREQ,
    writer=writer_smoke,
    verbose=1
)

writer_smoke.close()

# Validate basic functionality
print(f"\n{'='*60}")
print("SMOKE VALIDATION")
print(f"{'='*60}")
print(f"  Episodes completed: {len(result_smoke['episode_rewards'])}")
print(f"  Evaluations done:   {len(result_smoke['eval_history'])}")
if result_smoke['eval_history']:
    last_eval = result_smoke['eval_history'][-1]
    print(f"  Last eval return:   {last_eval['mean_return']:+.2f}%")
    print(f"  Last eval Sharpe:   {last_eval['sharpe']:.3f}")
    print(f"  Position fraction:  {last_eval['position_frac']:.1%}")
print(f"{'='*60}")
del result_smoke
print("✓ Smoke test passed")

## Hiperparàmetres optimitzables (15 total)

### From reference FeUdal Networks

| Parameter | Type | Range | Description |
|-----------|------|-------|-------------|
| `lr` | log-float | [1e-5, 1e-3] | RMSprop learning rate |
| `hidden_dim_manager` | cat | [64, 128, 256] | d: manager/perception output dim |
| `hidden_dim_worker` | cat | [8, 16, 24, 32] | k: worker hidden dim |
| `time_horizon` | int | [4, 24] | c: manager temporal horizon |
| `dilation` | int | [4, 24] | r: DilatedLSTM dilation level |
| `gamma_w` | float | [0.95, 0.999] | Worker discount factor |
| `gamma_m` | float | [0.99, 0.9999] | Manager discount factor |
| `alpha` | float | [0.1, 1.0] | Intrinsic reward coefficient |
| `entropy_coef` | log-float | [0.001, 0.1] | Entropy bonus weight |
| `num_steps` | cat | [100, 200, 400, 800] | Rollout length before update |
| `grad_clip` | float | [1.0, 10.0] | Gradient clipping norm |
| `eps` | log-float | [1e-7, 1e-3] | Random goal exploration probability |

### Custom (perception + environment)

| Parameter | Type | Range | Description |
|-----------|------|-------|-------------|
| `perception_hidden` | cat | [64, 128, 256] | MLP intermediate hidden dim |
| `hold_penalty` | float | [-0.1, -1e-5] | Env: FLAT+HOLD penalty |
| `realize_bonus_scale` | float | [0.1, 2.0] | Env: close bonus scale |

In [ ]:
# CEL·LA 10 — OPTUNA PHASE 1: TPE

def _compute_objective(eval_history, key='best_return'):
    """Compute Optuna objective from evaluation history."""
    if not eval_history:
        return float('-inf')
    if key == 'sharpe_final':
        return float(np.mean([e['sharpe'] for e in eval_history[-3:]]))
    return float(max(e['mean_return'] for e in eval_history))


def make_objective(df_tr, df_vl, prefix='TPE'):
    """Create Optuna objective function."""
    def objective(trial: optuna.Trial) -> float:
        hp = {
            'lr':                trial.suggest_float('lr', 1e-5, 1e-3, log=True),
            'hidden_dim_manager': trial.suggest_categorical('hidden_dim_manager', [64, 128, 256]),
            'hidden_dim_worker':  trial.suggest_categorical('hidden_dim_worker', [8, 16, 24, 32]),
            'time_horizon':       trial.suggest_categorical('time_horizon',[2, 4, 12, 24, 48]),
            'dilation':           trial.suggest_int('dilation', 4, 24),
            'gamma_w':            trial.suggest_float('gamma_w', 0.95, 0.999),
            'gamma_m':            trial.suggest_float('gamma_m', 0.99, 0.9999),
            'alpha':              trial.suggest_float('alpha', 0.1, 1.0),
            'entropy_coef':       trial.suggest_float('entropy_coef', 0.001, 0.1, log=True),
            'num_steps':          trial.suggest_categorical('num_steps', [100, 200, 400, 800]),
            'grad_clip':          trial.suggest_float('grad_clip', 1.0, 10.0),
            'eps':                trial.suggest_float('eps', 1e-7, 1e-3, log=True),
            'perception_hidden':  trial.suggest_categorical('perception_hidden', [64, 128, 256]),
            'hold_penalty':       trial.suggest_float('hold_penalty', -0.1, -1e-5),
            'realize_bonus_scale': trial.suggest_float('realize_bonus_scale', 0.1, 2.0),
            # Fixed
            'num_workers':        DEFAULT_HP['num_workers'],
        }

        run_name = f'{prefix}_{trial.number:03d}'
        writer_t = SummaryWriter(log_dir=str(RUNS_DIR / 'optuna' / run_name))

        try:
            result = train_feudal(
                hp=hp,
                max_steps=MAX_STEPS,
                eval_freq=EVAL_FREQ,
                df_tr=df_tr, df_vl=df_vl,
                trial=trial,
                writer=writer_t,
                verbose=0
            )
            obj_value = _compute_objective(result['eval_history'], OPTUNA_OBJECTIVE)

            # Check position fraction (avoid hold-only policies)
            if result['eval_history']:
                last_eval = result['eval_history'][-1]
                if last_eval['position_frac'] < 0.20:
                    raise optuna.TrialPruned(
                        f"position_frac={last_eval['position_frac']:.1%}")

        except optuna.TrialPruned:
            raise
        except Exception as e:
            print(f"Trial {trial.number} failed: {e}")
            return float('-inf')
        finally:
            writer_t.close()

        return obj_value

    return objective


# Create and run study
_p1_db = f"sqlite:///{OPTUNA_DB_DIR}/hrl_import_tpe.db"
_n_evals = MAX_STEPS // EVAL_FREQ
_warmup = max(1, int(_n_evals * 0.3))

study = optuna.create_study(
    direction='maximize',
    sampler=optuna.samplers.TPESampler(seed=SEED, n_startup_trials=N_STARTUP),
    pruner=optuna.pruners.MedianPruner(n_startup_trials=N_STARTUP, n_warmup_steps=_warmup),
    study_name=f'hrl_import_{NB_TAG}_tpe',
    storage=_p1_db,
    load_if_exists=True,
)

_completed = len([t for t in study.trials if t.state.name == 'COMPLETE'])
_remaining = max(0, N_TRIALS - _completed)
print(f'Optuna Phase 1 (TPE): {N_TRIALS} trials x {MAX_STEPS:,} steps')
print(f'  Completed: {_completed} | Remaining: {_remaining}')

tracker_optuna = EmissionsTracker(
    project_name='hrl_import_optuna', output_dir=str(CODECARBON_DIR), log_level='error'
)
tracker_optuna.start()
if _remaining > 0:
    study.optimize(make_objective(df_train, df_val),
                   n_trials=_remaining, show_progress_bar=True)
emissions_optuna = tracker_optuna.stop()
print(f'Emissions optuna: {emissions_optuna * 1000:.2f} g CO2eq')

# Extract best params
try:
    best_params = dict(study.best_params)
    best_value = study.best_value
except ValueError:
    best_params = DEFAULT_HP.copy()
    best_value = float('nan')
    print("WARNING: No completed trials. Using default params.")

# Add fixed params
best_params['num_workers'] = DEFAULT_HP['num_workers']

# Save
with open(OPTUNA_P1_JSON, 'w') as f:
    json.dump({'best_params': best_params, 'best_value': best_value}, f, indent=2)

print(f"\nBest Return (val): {best_value:+.4f}%")
print("Best params:")
for k, v in sorted(best_params.items()):
    print(f"  {k:<24}: {v}")

In [ ]:
# CEL·LA 11 — OPTUNA VISUALIZATION

# Optimization history
fig, ax = plt.subplots(figsize=(10, 4))
trials_complete = [t for t in study.trials if t.value is not None]
if trials_complete:
    ax.plot([t.number for t in trials_complete],
            [t.value for t in trials_complete], 'o-', alpha=0.7)
    ax.set_xlabel('Trial')
    ax.set_ylabel('Objective (Return %)')
    ax.set_title('Optuna TPE Optimization History')
    ax.axhline(best_value, color='r', linestyle='--', label=f'Best: {best_value:.2f}%')
    ax.legend()
    plt.tight_layout()
    plt.savefig(RESULTS / 'optuna_p1_history.png', dpi=120, bbox_inches='tight')
    plt.show()
else:
    print("No completed trials to plot.")

# Parameter importance
try:
    from optuna.importance import get_param_importances
    importances = get_param_importances(study)
    print("\nParameter importance:")
    for k, v in sorted(importances.items(), key=lambda x: -x[1])[:10]:
        print(f"  {k:<24}: {v:.4f}")
except Exception as e:
    print(f"Could not compute parameter importance: {e}")

In [ ]:
# CEL·LA 12 — FINAL TRAINING

FINAL_STEPS = 3_000_000 if not SMOKE else 5_000

writer_final = SummaryWriter(log_dir=str(RUNS_DIR / 'final'))

tracker_final = EmissionsTracker(
    project_name='hrl_import_final', output_dir=str(CODECARBON_DIR), log_level='error'
)
tracker_final.start()
result_final = train_feudal(
    hp=best_params,
    max_steps=FINAL_STEPS,
    eval_freq=EVAL_FREQ,
    writer=writer_final,
    verbose=1
)
emissions_final = tracker_final.stop()

writer_final.close()

# Save model
torch.save({
    'model': result_final['feudalnet'].state_dict(),
    'hp': best_params,
    'eval_history': result_final['eval_history'],
}, RESULTS / 'feudalnet_final.pt')

print(f"\n{'='*60}")
print("FINAL TRAINING COMPLETE")
print(f"{'='*60}")
if result_final['eval_history']:
    best_eval = max(result_final['eval_history'], key=lambda e: e['mean_return'])
    print(f"  Best validation return: {best_eval['mean_return']:+.2f}%")
    print(f"  Best Sharpe:            {best_eval['sharpe']:.3f}")
    print(f"  Position fraction:      {best_eval['position_frac']:.1%}")
print(f"  Emissions final:        {emissions_final * 1000:.2f} g CO2eq")
print(f"  Model saved to: {RESULTS / 'feudalnet_final.pt'}")
print(f"{'='*60}")

In [ ]:
# CEL·LA 13 — TRAINING CURVES

fig, axes = plt.subplots(2, 2, figsize=(14, 8))

# Episode rewards
if result_final['episode_rewards']:
    rewards = result_final['episode_rewards']
    axes[0,0].plot(rewards, alpha=0.3, color='blue')
    # Rolling mean
    window = min(50, len(rewards)//4) if len(rewards) > 4 else 1
    if window > 1:
        rm = np.convolve(rewards, np.ones(window)/window, mode='valid')
        axes[0,0].plot(range(window-1, len(rewards)), rm, color='red', lw=2)
    axes[0,0].set_title('Episode Returns')
    axes[0,0].set_xlabel('Episode')
    axes[0,0].set_ylabel('Return (%)')

# Eval metrics over time
if result_final['eval_history']:
    evals = result_final['eval_history']
    eval_steps = [i * EVAL_FREQ for i in range(1, len(evals)+1)]
    
    axes[0,1].plot(eval_steps, [e['mean_return'] for e in evals], 'o-')
    axes[0,1].set_title('Validation Return')
    axes[0,1].set_ylabel('Return (%)')
    
    axes[1,0].plot(eval_steps, [e['sharpe'] for e in evals], 'o-', color='green')
    axes[1,0].set_title('Validation Sharpe')
    axes[1,0].set_ylabel('Sharpe')
    
    axes[1,1].plot(eval_steps, [e['position_frac'] for e in evals], 'o-', color='orange')
    axes[1,1].set_title('Position Fraction')
    axes[1,1].set_ylabel('Fraction')

plt.tight_layout()
plt.savefig(RESULTS / 'training_curves.png', dpi=120, bbox_inches='tight')
plt.show()

In [ ]:
# CEL·LA 14 — TEST EVALUATION

feudalnet_final = result_final['feudalnet']

test_result = evaluate_feudal(feudalnet_final, df_test, best_params,
                              n_episodes=10 if not SMOKE else 2)

print("="*60)
print("TEST SET EVALUATION")
print("="*60)
print(f"  Mean Return:     {test_result['mean_return']:+.2f}% +/- {test_result['std_return']:.2f}%")
print(f"  Sharpe Ratio:    {test_result['sharpe']:.3f}")
print(f"  Max Drawdown:    {test_result['max_drawdown']:.2f}%")
print(f"  Mean Trades:     {test_result['mean_trades']:.1f}")
print(f"  Position Frac:   {test_result['position_frac']:.1%}")
print("="*60)

# ── Emissions summary ──
_em_opt = float(emissions_optuna) if 'emissions_optuna' in dir() else 0.0
_em_fin = float(emissions_final)  if 'emissions_final'  in dir() else 0.0
_em_total_g = (_em_opt + _em_fin) * 1000
print(f"\n  CO2 optuna:      {_em_opt * 1000:.2f} g CO2eq")
print(f"  CO2 final:       {_em_fin * 1000:.2f} g CO2eq")
print(f"  CO2 total:       {_em_total_g:.3f} g CO2eq")

# Save final metrics
metrics = {
    'best_params': best_params,
    'optuna_best_value': best_value,
    'test_metrics': test_result,
    'n_trials': N_TRIALS,
    'final_steps': FINAL_STEPS,
    'sustainability': {
        'emissions_optuna_kg': _em_opt,
        'emissions_final_kg':  _em_fin,
        'emissions_total_g':   _em_total_g,
    },
}
with open(RESULTS / 'final_metrics.json', 'w') as f:
    json.dump(metrics, f, indent=2, default=str)

print(f"\n✓ All results saved to: {RESULTS}")

## Verificació: walk-forward test i comparativa amb checkpoint

Comprovem que `FeudalAgentV9.load()` reprodueix exactament els mateixos resultats
que el model entrenat directament al notebook (`feudalnet_final`).

- **Walk-forward** amb `feudalnet_final` (embolicat en `FeudalAgentV9`)
- **Càrrega des de checkpoint** via `FeudalAgentV9.load()`
- **Comparativa de pesos i mètriques**

In [ ]:
# CEL·LA 15 — WALK-FORWARD AMB feudalnet_final (via FeudalAgentV9 wrapper)
# Protocol seqüencial (igual que walk_forward_eval_hrl en 05d/05s).

from src.rl.agents.feudal_agent_v9 import FeudalAgentV9, _FeudalNetV9
from src.rl.envs.crypto_market_env import MAX_EPISODE_STEPS

N_WF_EPISODES = 10 if not SMOKE else 2


def walk_forward_eval_v9(agent_v9, df, max_episodes=0, deterministic=True):
    ep_steps = MAX_EPISODE_STEPS
    env = make_env(df,
                   hold_penalty=best_params.get("hold_penalty", HOLD_PENALTY),
                   realize_bonus_scale=best_params.get("realize_bonus_scale", 0.7))
    n_rows = len(df)
    c = agent_v9.c
    all_equities, all_actions, all_prices, ep_returns = [], [], [], []
    all_trades, all_pos_sides = [], []
    start = 0
    agent_v9.eval()
    with torch.no_grad():
        while start + ep_steps + 1 < n_rows:
            if max_episodes > 0 and len(ep_returns) >= max_episodes:
                break
            obs, info = env.reset(options={"episode_start": start})
            ep_eq = [info["equity"]]
            done, step_ep, cur_goal = False, 0, None
            while not done:
                action, cur_goal = agent_v9.predict_step(
                    obs, step_ep, c, cur_goal, str(DEVICE), deterministic)
                obs, _, term, trunc, info = env.step(action)
                done = term or trunc
                step_ep += 1
                ep_eq.append(info["equity"])
                all_actions.append(action)
                all_prices.append(info.get("current_price", np.nan))
                all_pos_sides.append(info["pos_side"])
            all_equities.extend(ep_eq)
            ep_returns.append((ep_eq[-1] / ep_eq[0] - 1) * 100)
            all_trades.append(info["n_trades"])
            start += len(ep_eq) - 1
    env.close()
    agent_v9.train()
    eq_arr = np.array(all_equities, dtype=np.float64)
    n_pos = sum(1 for p in all_pos_sides if p != 0)
    act_names = ["HOLD", "LONG", "SHORT", "CLOSE"]
    return {
        "equities":   all_equities,
        "prices":     all_prices,
        "actions":    all_actions,
        "ep_returns": ep_returns,
        "metrics": {
            "total_return":  float((eq_arr[-1] / eq_arr[0] - 1) * 100) if len(eq_arr) > 1 else 0.0,
            "mean_return":   float(np.mean(ep_returns)) if ep_returns else 0.0,
            "std_return":    float(np.std(ep_returns)) if ep_returns else 0.0,
            "sharpe":        float(compute_sharpe(eq_arr)),
            "max_drawdown":  float(compute_max_drawdown(all_equities)),
            "n_episodes":    len(ep_returns),
            "n_steps":       len(all_actions),
            "mean_trades":   float(np.mean(all_trades)) if all_trades else 0.0,
            "position_frac": n_pos / max(len(all_pos_sides), 1),
            "action_dist_pct": {
                name: all_actions.count(i) / max(len(all_actions), 1) * 100
                for i, name in enumerate(act_names)
            },
        },
    }


# Embolicar feudalnet_final en FeudalAgentV9
net_wrap = _FeudalNetV9(
    obs_dim           = OBS_DIM,
    d                 = best_params["hidden_dim_manager"],
    k                 = best_params["hidden_dim_worker"],
    n_actions         = N_ACTIONS,
    c                 = best_params["time_horizon"],
    r                 = best_params["dilation"],
    perception_hidden = best_params["perception_hidden"],
    device            = str(DEVICE),
)
net_wrap.load_state_dict(feudalnet_final.state_dict())
net_wrap.eval().to(DEVICE)
agent_nb = FeudalAgentV9(net=net_wrap, c=best_params["time_horizon"])

print("Walk-forward seqüencial (feudalnet_final) ...")
wf_nb = walk_forward_eval_v9(agent_nb, df_test, max_episodes=N_WF_EPISODES)
m = wf_nb["metrics"]

print()
print("=" * 58)
print("  RESULTATS FeudalAgentV9 — TEST SPLIT (walk-forward)")
print("=" * 58)
print(f"  Retorn total acumulat : {m['total_return']:>+10.2f} %")
print(f"  Retorn mig / episodi  : {m['mean_return']:>+10.2f} +/- {m['std_return']:.2f} %")
print(f"  Sharpe ratio          : {m['sharpe']:>+10.4f}")
print(f"  Max drawdown          : {m['max_drawdown']:>10.2f} %")
print(f"  Episodis              : {m['n_episodes']:>10,}")
print(f"  Passos totals         : {m['n_steps']:>10,}")
print(f"  Trades mig/ep         : {m['mean_trades']:>10.1f}")
print(f"  Frac. posicio oberta  : {m['position_frac']:>10.1%}")
print("  Distribucio accions:")
for act_name, pct in m["action_dist_pct"].items():
    bar = "#" * int(pct / 2)
    print(f"    {act_name:<6}: {pct:>6.2f} %  {bar}")
print("=" * 58)
p_arr = np.array(wf_nb["prices"], dtype=np.float64)
p_arr = p_arr[~np.isnan(p_arr)]
if len(p_arr) > 0:
    bah = (p_arr[-1] / p_arr[0] - 1) * 100
    print(f"  Buy-and-Hold ref: {bah:>+8.2f}%")
    print(f"  Alpha vs B&H:     {m['total_return'] - bah:>+8.2f}%")
print()
print("Walk-forward notebook OK")

In [ ]:
# CEL·LA 16 — VERIFICACIÓ: src.FeudalAgentV9.load() ≡ feudalnet_final

CKPT_PATH = RESULTS / 'feudalnet_final.pt'

# ── Càrrega des de src ──────────────────────────────────────────────────────────
agent_src = FeudalAgentV9.load(str(CKPT_PATH), device=str(DEVICE))
print(f'Carregat des de src:')
print(f'  obs_dim={OBS_DIM}  d={agent_src._net.d}  k={agent_src._net.k}'
      f'  n_actions={agent_src._net.n_actions}  c={agent_src.c}  r={agent_src._net.r}')
total_params = sum(p.numel() for p in agent_src._net.parameters())
print(f'  Paràmetres totals: {total_params:,}')

# ── Verificació de pesos ────────────────────────────────────────────────────────
n_checked = 0
for key, w_orig in feudalnet_final.state_dict().items():
    w_load = agent_src._net.state_dict()[key]
    assert torch.allclose(w_orig.cpu(), w_load.cpu()), f'Divergència en capa: {key}'
    n_checked += 1
print(f'Pesos idèntics a feudalnet_final ({n_checked} tensors verificats) ✓')

# ── Walk-forward amb l'agent carregat des de src ────────────────────────────────
print()
print(f'Walk-forward amb src.FeudalAgentV9.load()  [{N_WF_EPISODES} episodis] ...')
wf_src = walk_forward_eval_v9(agent_src, df_test, max_episodes=N_WF_EPISODES)

# ── Comparació numèrica ─────────────────────────────────────────────────────────
m_nb  = wf_nb['metrics']
m_src = wf_src['metrics']

print()
print(f"{'='*66}")
print(f"  {'Mètrica':<22} {'feudalnet_final':>18} {'src.load()':>18}  OK")
print(f"  {'-'*64}")
for key in ['mean_return', 'sharpe', 'max_drawdown', 'mean_trades', 'position_frac']:
    v1, v2 = m_nb[key], m_src[key]
    ok_sym = '✓' if abs(v1 - v2) < 1e-4 else '✗'
    print(f'  {key:<22} {v1:>+18.4f} {v2:>+18.4f}  {ok_sym}')
print(f"{'='*66}")

# ── Verificació d'accions idèntiques ──────────────────────────────────────────
acts_nb  = wf_nb['actions']
acts_src = wf_src['actions']
if acts_nb == acts_src:
    print("Seqüència d'accions idèntica ✓")
else:
    diff_count = sum(a != b for a, b in zip(acts_nb, acts_src))
    print(f"ATENCIÓ: {diff_count} accions difereixen (possible no-determinisme)")

print()
print('NB07d — Verificació src.FeudalAgentV9 OK')